# SPE11B Task 1: Full-feature temporal surrogate

This notebook builds a small benchmark surrogate for SPE11B spatial maps.
The model learns how to move a full spatial state forward in time, first with one-step prediction and then with autoregressive rollout.

## Benchmark goal

Given an observed state at time $t$, predict the future state at a chosen horizon $T$.
In practice, we start from one snapshot, predict the next snapshot, and repeat the same learned map until we reach the target time.

## Dataset snapshot

- The published benchmark contains 34 participant submissions.
- Each snapshot is stored as a CSV file on a fixed grid of 100,800 points.
- Every row contains one grid location and the associated physical fields.
- The grid coordinates stay fixed through time, so the task is temporal prediction on a fixed spatial mesh.

## Features

Each row contains two coordinate columns and eight state variables:

- `x [m]`, `z [m]`: fixed spatial coordinates
- `pressure [Pa]`
- `saturation [-]`
- `mCO2 [-]`
- `mH2O [-]`
- `rhoG [kg/m3]`
- `rhoL [kg/m3]`
- `tmCO2 [kg]`
- `temp [°C]`

The surrogate predicts the state variables forward in time and keeps the coordinates fixed from the template CSV.

## Notation

Let $s_t$ denote the full spatial state at time $t$.
In this notebook, one step of the surrogate is written as

$$
s_{t+\Delta t} = f_{\theta}(s_t, \Delta t)
$$

where $\Delta t$ is the time gap between two snapshots and $f_{\theta}$ is a learned regression map.
When we apply the same map repeatedly, we get an autoregressive rollout:

$$
s_{t_{k+1}} = f_{\theta}(s_{t_k}, t_{k+1} - t_k)
$$

This notebook uses all state variables except the coordinates as model targets, while the coordinates are kept fixed by the template CSV during export.

## What this notebook shows

- how to discover and index the raw SPE11B files
- how to build a small transition dataset for training
- how to fit a tiny ridge-regression baseline
- how to export predictions in benchmark CSV format
- how to evaluate both one-step and autoregressive rollouts


In [ ]:
from pathlib import Path
import subprocess
import sys
from IPython.display import Image, display
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
import numpy as np

def find_repo_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / 'scripts' / 'download_spe11b.py').exists():
            return candidate
    raise FileNotFoundError('Could not locate the repository root.')

repo_root = find_repo_root()
if str(repo_root / 'src') not in sys.path:
    sys.path.insert(0, str(repo_root / 'src'))

from ml4gcs.data import (
    ParticipantSeries,
    SpatialMapSnapshot,
    SpatialMapTransitionDataset,
    build_spatial_map_index,
    build_spatial_map_transition_batch,
    export_next_step_timeline_data,
    fit_feature_normalizer,
    save_spatial_map_csv_data,
)

data_root = repo_root / 'spe11b'
if not data_root.exists():
    if Path('/home/jovyan/shared_folder').exists():
        data_root = Path('/home/jovyan/shared_folder/data/spe11b')
    else:
        subprocess.run([sys.executable, str(repo_root / 'scripts' / 'download_spe11b.py')], check=True)

## Data setup
We index the unpacked SPE11B files, pick one participant for training, and keep another participant aside for export and inspection.
In this benchmark pass, training is restricted to the first 50 years of one participant so the notebook stays lightweight and easy to explore.


In [ ]:
index = build_spatial_map_index(data_root)
participants = sorted({ref.participant for ref in index})
print(f'Found {len(index)} spatial-map files across {len(participants)} participants.')
print('First participant:', participants[0])
print('Last participant:', participants[-1])

# Train on one participant from the first 50 years only.
train_participant = participants[0]
export_participant = participants[3]
train_dataset = SpatialMapTransitionDataset(
    data_root,
    participants=[train_participant],
    time_years_range=(0.0, 50.0),
)
train_transitions = tuple(train_dataset)
train_cutoff_time = max(transition.target_time_years for transition in train_transitions)

print('train participant:', train_participant)
print('train transitions:', len(train_transitions))
print('train horizon ends at:', train_cutoff_time)
print('export participant:', export_participant)


## Build the training set
The transition dataset keeps the benchmark logic in one place: it groups files by participant, orders them by time, and exposes consecutive snapshot pairs.


In [ ]:
train_batch = build_spatial_map_transition_batch(train_transitions, rows_per_transition=None, seed=42)
state_columns = list(train_batch.target_columns)
state_idx = [train_transitions[0].input_snapshot.columns.index(name) for name in state_columns]

x_stats = fit_feature_normalizer(train_batch.X[:, :-1])
y_stats = fit_feature_normalizer(train_batch.Y)
delta_stats = fit_feature_normalizer(train_batch.delta_time_years)

X_train = np.hstack([
    x_stats.normalize(train_batch.X[:, :-1]),
    delta_stats.normalize(train_batch.delta_time_years),
])
Y_train = y_stats.normalize(train_batch.Y)

print('train transitions:', len(train_transitions))
print('training matrix shape:', X_train.shape)
print('training targets shape:', Y_train.shape)
print('feature columns:', list(train_batch.feature_columns))
print('target columns:', list(train_batch.target_columns))


## Fit a tiny baseline
A small ridge regressor is enough for a first benchmark pass and keeps the notebook fast to run.


In [ ]:
def fit_ridge_regression(X, Y, ridge=1.0):
    X = np.asarray(X, dtype=np.float64)
    Y = np.asarray(Y, dtype=np.float64)
    X_aug = np.hstack([np.ones((X.shape[0], 1)), X])
    xtx = X_aug.T @ X_aug
    xty = X_aug.T @ Y
    xtx += ridge * np.eye(xtx.shape[0])
    return np.linalg.solve(xtx, xty)
def predict_ridge_regression(weights, X):
    X = np.asarray(X, dtype=np.float64)
    X_aug = np.hstack([np.ones((X.shape[0], 1)), X])
    return X_aug @ weights
weights = fit_ridge_regression(X_train, Y_train, ridge=1.0)
print('weights shape:', weights.shape)
print('parameter count:', weights.size)


## Export predictions
The model writes benchmark-style CSV files so the outputs can be compared directly to participant submissions.


In [ ]:
def predict_next_snapshot_data(input_snapshot, target_snapshot):
    delta = np.full(
        (input_snapshot.data.shape[0], 1),
        float(target_snapshot.time_years - input_snapshot.time_years),
        dtype=np.float64,
    )
    x = input_snapshot.data[:, state_idx]
    x_model = np.hstack([x_stats.normalize(x), delta_stats.normalize(delta)])
    pred_norm = predict_ridge_regression(weights, x_model)
    pred_physical = y_stats.denormalize(pred_norm)
    pred = target_snapshot.data.copy()
    pred[:, state_idx] = pred_physical
    return pred

export_refs = [ref for ref in index if ref.participant == export_participant]
export_series = ParticipantSeries(
    participant=export_participant,
    snapshots=tuple(ref.load() for ref in export_refs),
)
export_root = repo_root / 'reports' / 'predictions' / 'ridge_next_step_full_features'
written = export_next_step_timeline_data(export_series, predict_next_snapshot_data, export_root)
print('export participant:', export_series.participant)
print('written files:', len(written))
print('first output:', written[0])


## Evaluation

We first compute the benchmark summaries, then plot the global errors and the full timeline separately.


In [ ]:
field_names = list(export_series.snapshots[0].columns)
physical_fields = list(train_batch.target_columns)

def masked_rmse(pred, target):
    pred = np.asarray(pred, dtype=np.float64)
    target = np.asarray(target, dtype=np.float64)
    mask = np.isfinite(pred) & np.isfinite(target)
    if not np.any(mask):
        return np.nan
    diff = pred[mask] - target[mask]
    return float(np.sqrt(np.mean(diff * diff)))

def masked_relative_l2(pred, target):
    pred = np.asarray(pred, dtype=np.float64)
    target = np.asarray(target, dtype=np.float64)
    mask = np.isfinite(pred) & np.isfinite(target)
    if not np.any(mask):
        return np.nan
    diff = pred[mask] - target[mask]
    denom = np.sqrt(np.sum(target[mask] * target[mask]))
    if denom == 0.0:
        return float(np.sqrt(np.sum(diff * diff)))
    return float(np.sqrt(np.sum(diff * diff)) / denom)

def masked_normalized_rmse(pred, target):
    target = np.asarray(target, dtype=np.float64)
    scale = float(np.nanstd(target))
    rmse = masked_rmse(pred, target)
    if not np.isfinite(rmse):
        return np.nan
    return rmse / scale if scale > 0 else rmse

export_summaries = []
for input_snapshot, target_snapshot in zip(export_series.snapshots[:-1], export_series.snapshots[1:], strict=True):
    prediction = predict_next_snapshot_data(input_snapshot, target_snapshot)
    row_summary = {'target_time_years': target_snapshot.time_years}

    pressure_idx = field_names.index('pressure [Pa]')
    target_pressure = target_snapshot.data[:, pressure_idx]
    pred_pressure = prediction[:, pressure_idx]
    row_summary['pressure [Pa]'] = masked_relative_l2(pred_pressure, target_pressure)

    for field_name in physical_fields:
        col_idx = field_names.index(field_name)
        target_values = target_snapshot.data[:, col_idx]
        pred_values = prediction[:, col_idx]
        row_summary[field_name] = masked_normalized_rmse(pred_values, target_values)

    export_summaries.append(row_summary)

export_summaries = sorted(export_summaries, key=lambda item: item['target_time_years'])
times = np.array([summary['target_time_years'] for summary in export_summaries], dtype=float)
pressure_over_time = np.array([summary['pressure [Pa]'] for summary in export_summaries], dtype=float)
feature_traces = {name: np.array([summary[name] for summary in export_summaries], dtype=float) for name in physical_fields}
feature_order = sorted(feature_traces.keys(), key=lambda name: float(np.nanstd(feature_traces[name])), reverse=True)[:4]
feature_traces = {name: feature_traces[name] for name in feature_order}

valid_pressure = np.isfinite(pressure_over_time)
global_pressure_rmse = float(np.mean(pressure_over_time[valid_pressure])) if np.any(valid_pressure) else np.nan
global_feature_rmse = {
    name: float(np.mean(values[np.isfinite(values)])) if np.any(np.isfinite(values)) else np.nan
    for name, values in feature_traces.items()
}
train_cutoff_time = max(transition.target_time_years for transition in train_transitions)
injection_periods = [
    (0.0, 25.0, 'well 1 injecting'),
    (25.0, 50.0, 'well 2 injecting'),
]
phase_colors = {
    'well 1 injecting': '#C44E52',
    'well 2 injecting': '#4C72B0',
    'post-injection': '#55A868',
}
phase_patches = [Patch(facecolor=color, edgecolor=color, alpha=0.12, label=label) for label, color in phase_colors.items()]


### Global summary

Pressure is reported with relative $L^2$, while the remaining fields are normalized by the target standard deviation.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13.5, 4.8), constrained_layout=True)
axes[0].bar(['pressure [Pa]'], [global_pressure_rmse], color='#C44E52', width=0.6)
axes[0].set_ylabel('Relative L2 error')
axes[0].set_title(f'Global pressure relative L2 on {export_series.participant}')
axes[0].set_ylim(bottom=0)

axes[1].bar(list(global_feature_rmse.keys()), list(global_feature_rmse.values()), color='#4C72B0')
axes[1].set_yscale('log')
axes[1].set_ylabel('Normalized RMSE')
axes[1].set_title('Global feature-level error (RMSE / target std)')
axes[1].tick_params(axis='x', rotation=45)
axes[1].set_ylim(bottom=1e-8)
summary_path = repo_root / 'reports' / 'figures' / 'spe11b_global_error.png'
summary_path.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(summary_path, bbox_inches='tight')
plt.close(fig)
display(Image(filename=str(summary_path)))


### Timeline error

The next plot shows how the error evolves across the full held-out timeline, with the training window marked by a vertical line and the injection phases shaded in the background.


In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(13.5, 8.0), sharex=True, constrained_layout=True)

ax = axes[0]
if np.any(np.isfinite(pressure_over_time)):
    ax.plot(times, pressure_over_time, marker='o', linewidth=2.2, label='pressure relative L2', color='#C44E52')
for start, end, label in injection_periods:
    ax.axvspan(start, end, color=phase_colors[label], alpha=0.12)
ax.axvline(train_cutoff_time, color='black', linestyle='--', linewidth=1.5, alpha=0.8, label='train cutoff')
ax.set_ylabel('Pressure relative L2')
ax.set_title(f'Error evolution through time on {export_series.participant}')
line_handles, line_labels = ax.get_legend_handles_labels()
ax.legend(handles=line_handles + phase_patches, labels=line_labels + [p.get_label() for p in phase_patches], ncol=2, fontsize=8, frameon=True)

ax = axes[1]
for field_name, values in feature_traces.items():
    if np.any(np.isfinite(values)):
        ax.plot(times, values, marker='o', linewidth=1.6, alpha=0.85, label=field_name)
for start, end, label in injection_periods:
    ax.axvspan(start, end, color=phase_colors[label], alpha=0.08)
ax.axvline(train_cutoff_time, color='black', linestyle='--', linewidth=1.5, alpha=0.8)
ax.set_xlabel('target time [years]')
ax.set_ylabel('Normalized RMSE')
ax.set_yscale('log')
ax.set_ylim(bottom=1e-8)
line_handles, line_labels = ax.get_legend_handles_labels()
ax.legend(handles=line_handles + phase_patches, labels=line_labels + [p.get_label() for p in phase_patches], ncol=2, fontsize=8, frameon=True)
axes[1].set_xlim(min(times), max(times))
evolution_path = repo_root / 'reports' / 'figures' / 'spe11b_error_evolution.png'
evolution_path.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(evolution_path, bbox_inches='tight')
plt.close(fig)
display(Image(filename=str(evolution_path)))


## Autoregressive rollout

An autoregressive model predicts one step ahead and then feeds its own prediction back as the next input. In this benchmark, that lets us start from any snapshot and keep moving forward in time until the desired horizon.


In [9]:
def predict_autoregressive_data(current_snapshot, target_snapshot):
    delta = np.full(
        (current_snapshot.data.shape[0], 1),
        float(target_snapshot.time_years - current_snapshot.time_years),
        dtype=np.float64,
    )
    x = current_snapshot.data[:, state_idx]
    x_model = np.hstack([x_stats.normalize(x), delta_stats.normalize(delta)])
    pred_norm = predict_ridge_regression(weights, x_model)
    pred_physical = y_stats.denormalize(pred_norm)
    pred = target_snapshot.data.copy()
    pred[:, state_idx] = pred_physical
    return pred

ar_root = repo_root / 'reports' / 'predictions' / 'ridge_autoregressive_full_features'
ar_root.mkdir(parents=True, exist_ok=True)
ar_outputs = []
ar_summaries = []
ar_feature_names = [name for name in field_names if name not in ['x [m]', 'z [m]', 'pressure [Pa]']]
ar_feature_names = [name for name in ar_feature_names if name in physical_fields]
ar_feature_traces = {name: [] for name in ar_feature_names}

current_snapshot = export_series.snapshots[0]
for target_snapshot in export_series.snapshots[1:]:
    predicted_data = predict_autoregressive_data(current_snapshot, target_snapshot)
    output_path = ar_root / export_series.participant / target_snapshot.path.name
    save_spatial_map_csv_data(target_snapshot, output_path, predicted_data)
    predicted_snapshot = SpatialMapSnapshot(
        participant=target_snapshot.participant,
        time_years=target_snapshot.time_years,
        path=output_path,
        columns=target_snapshot.columns,
        data=predicted_data,
    )
    ar_outputs.append(output_path)

    pressure_idx = target_snapshot.columns.index('pressure [Pa]')
    ar_summaries.append({
        'target_time_years': target_snapshot.time_years,
        'pressure [Pa]': masked_relative_l2(predicted_data[:, pressure_idx], target_snapshot.data[:, pressure_idx]),
    })
    for field_name in ar_feature_names:
        col_idx = target_snapshot.columns.index(field_name)
        ar_feature_traces[field_name].append(masked_normalized_rmse(predicted_data[:, col_idx], target_snapshot.data[:, col_idx]))
    current_snapshot = predicted_snapshot

ar_summary_times = np.array([item['target_time_years'] for item in ar_summaries], dtype=float)
ar_pressure = np.array([item['pressure [Pa]'] for item in ar_summaries], dtype=float)
ar_feature_traces = {name: np.array(values, dtype=float) for name, values in ar_feature_traces.items()}
ar_feature_order = sorted(ar_feature_traces.keys(), key=lambda name: float(np.nanstd(ar_feature_traces[name])), reverse=True)[:4]
ar_feature_traces = {name: ar_feature_traces[name] for name in ar_feature_order}
ar_pressure_mean = float(np.mean(ar_pressure[np.isfinite(ar_pressure)])) if np.any(np.isfinite(ar_pressure)) else np.nan


### Autoregressive errors

The pressure curve is shown separately, and the other fields are plotted on a shared normalized scale so small effects stay visible.


In [ ]:
fig, ax = plt.subplots(figsize=(13.5, 4.8), constrained_layout=True)
ax.plot(ar_summary_times, ar_pressure, marker='o', linewidth=2.0, color='#C44E52', label='autoregressive pressure relative L2')
ax.axvline(train_cutoff_time, color='black', linestyle='--', linewidth=1.5, alpha=0.8, label='train cutoff')
ax.set_xlabel('target time [years]')
ax.set_ylabel('Pressure relative L2')
ax.set_title(f'Autoregressive rollout on {export_series.participant}')
ax.legend(frameon=True)
ar_plot_path = repo_root / 'reports' / 'figures' / 'spe11b_autoregressive_pressure.png'
fig.savefig(ar_plot_path, bbox_inches='tight')
plt.close(fig)
display(Image(filename=str(ar_plot_path)))

fig, ax = plt.subplots(figsize=(13.5, 4.8), constrained_layout=True)
for field_name, values in ar_feature_traces.items():
    ax.plot(ar_summary_times, values, marker='o', linewidth=1.8, label=field_name)
ax.axvline(train_cutoff_time, color='black', linestyle='--', linewidth=1.5, alpha=0.8, label='train cutoff')
ax.set_xlabel('target time [years]')
ax.set_ylabel('Normalized RMSE')
ax.set_yscale('log')
ax.set_ylim(bottom=1e-8)
ax.set_title(f'Autoregressive feature error evolution on {export_series.participant}')
ax.legend(frameon=True, ncol=2, fontsize=8)
ar_feature_plot_path = repo_root / 'reports' / 'figures' / 'spe11b_autoregressive_features.png'
fig.savefig(ar_feature_plot_path, bbox_inches='tight')
plt.close(fig)
display(Image(filename=str(ar_feature_plot_path)))


### 1000-year pressure comparison

The final image compares the autoregressive pressure field at the last time step against the target and shows the absolute difference.


In [ ]:
final_target_snapshot = export_series.snapshots[-1]
final_predicted_snapshot = current_snapshot
final_target_pressure = final_target_snapshot.reshape_field('pressure [Pa]')
final_pred_pressure = final_predicted_snapshot.reshape_field('pressure [Pa]')
final_pressure_diff = np.abs(final_pred_pressure - final_target_pressure)

fig, axes = plt.subplots(1, 3, figsize=(16, 4.8), sharex=True, sharey=True, constrained_layout=True)
im0 = axes[0].imshow(final_target_pressure, origin='lower', aspect='auto', cmap='viridis')
axes[0].set_title(f'Target pressure @ {final_target_snapshot.time_years:g} y')
im1 = axes[1].imshow(final_pred_pressure, origin='lower', aspect='auto', cmap='viridis')
axes[1].set_title('Autoregressive prediction')
im2 = axes[2].imshow(final_pressure_diff, origin='lower', aspect='auto', cmap='magma')
axes[2].set_title('Absolute difference')
for ax in axes:
    ax.set_xlabel('x index')
axes[0].set_ylabel('z index')
fig.colorbar(im2, ax=axes, shrink=0.85, label='|pressure difference|')
pressure_1000_path = repo_root / 'reports' / 'figures' / 'spe11b_autoregressive_pressure_1000y.png'
fig.savefig(pressure_1000_path, bbox_inches='tight')
plt.close(fig)
display(Image(filename=str(pressure_1000_path)))

print('autoregressive outputs:', len(ar_outputs))
print('mean autoregressive pressure relative L2:', ar_pressure_mean)
print('first output:', ar_outputs[0])
print('last output:', ar_outputs[-1])
